# Baseline eval of the unpruned teacher: general knowledge

Part of plan.md step 1.2's baseline eval suite (`Qwen3-30B-A3B-Base` as-is --
the number every pruned candidate in Phase 2/4 gets compared to), covering
the **tracked, not-optimized-for** tier: MMLU (5-shot), ARC-Challenge
(25-shot), Winogrande (5-shot), full test sets. Shot counts are pinned in
`bonsai.eval.general_knowledge.NUM_FEWSHOT` since none of the three tasks'
own harness configs set one -- left at the library default they'd silently
fall back to 0-shot, making the numbers incomparable to the papers' tables.

Run independently from HumanEval-TS and MBPP-TS (each gets its own
notebook) so a failure or a long run in one doesn't block or hide the
others. Called directly as a library (`lm_eval.simple_evaluate`), not via
the CLI -- logs stream into the cell the same way a shell magic would, and
the returned dict is what gets saved, no separate `--output_path` plumbing
needed.

These are log-likelihood-scoring tasks, not generation -- far cheaper per
item than HumanEval-TS/MBPP-TS despite MMLU alone being ~14k questions.

Needs a rented GPU (1x 80GB, per plan.md's sizing table) with
`WITH_EVAL=1 ./tools/setup_instance.sh` (installs the `eval` extra; this
notebook doesn't need Node.js/`tsc`, only the code-gen tasks do).

In [1]:
import json

from bonsai.config import load_config
from bonsai.eval.general_knowledge import run_general_knowledge
from bonsai.tracking import log_metric, new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
run_dir = new_run("baseline-eval")
save_config_snapshot(run_dir, config)

print(f"model: {config.model.repo_id}")
print(f"run dir: {run_dir}")

model: Qwen/Qwen3-30B-A3B-Base
run dir: runs/20260918T171406Z-30948c-baseline-eval


In [2]:
general_knowledge = run_general_knowledge(
    config.model.repo_id, revision=config.model.revision, dtype="bfloat16", batch_size="auto"
)

for task, result in general_knowledge.items():
    print(task, result["results"][task])

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

abstract_algebra/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 9.96kB            

abstract_algebra/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

abstract_algebra/validation-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 3.73kB            

... (6,016 outputs left out)

Running loglikelihood requests:  99%|█████████▊| 2497/2534 [00:23<00:00, 144.30it/s]

Running loglikelihood requests: 100%|██████████| 2534/2534 [00:23<00:00, 106.08it/s]


fatal: not a git repository (or any of the parent directories): .git


mmlu {'alias': 'mmlu', 'name': 'mmlu', 'sample_len': 14042, 'acc,none': 0.8101410055547643, 'acc_stderr,none': 0.0031497324534049006, 'sample_count': {'acc,none': 14042}}
arc_challenge {'name': 'arc_challenge', 'alias': 'arc_challenge', 'sample_len': 1172, 'acc,none': 0.6535836177474402, 'acc_stderr,none': 0.013905011180063256, 'acc_norm,none': 0.6749146757679181, 'acc_norm_stderr,none': 0.013688147309729217}
winogrande {'name': 'winogrande', 'alias': 'winogrande', 'sample_len': 1267, 'acc,none': 0.7797947908445146, 'acc_stderr,none': 0.011646276755089623}


In [3]:
(run_dir / "general-knowledge.json").write_text(
    json.dumps(general_knowledge, indent=2, default=str)
)

for task, result in general_knowledge.items():
    log_metric(run_dir, task=task, **result["results"][task])